In [1]:
#!pip install openai

In [2]:
#import os, openai
import torch, json, re, os, ast
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
proxy = "http://cache.ha.aaaa-bbbbb.fr:xxxx/"

os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

In [4]:
print(os.listdir("/home/partage/"))

['Mistral-7B-Instruct-v0.2', 'Llama-2-7b-chat-hf', 'Llama-3.1-8B-Instruct', 'Llama-3.2-1B-Instruct', 'Qwen', 'test.txt', 'Mistral-7B-Instruct-v0.3', 'Meta-Llama-3-8B-Instruct', '.Trash-1001', 'Olmo-3-7B-Instruct', 'microsoft', 'Llama-2-13b-chat-hf', 'google', 'Llama-3.2-3B-Instruct']


In [5]:
model_path = "/home/partage/Llama-3.1-8B-Instruct"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [8]:
def llm(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            # temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [9]:
with open("../data/json/rewrites.json","r" , encoding='utf-8') as file:
    data = json.load(file)

In [10]:
def evaluation(source, rewrite):
    prompt = f"""
You are an expert evaluator.

Think carefully, but DO NOT show your reasoning.

Return ONLY ONE result in this EXACT format:
[[factual_score, semantic_score]]

If the format is not EXACTLY [[X, Y]], the answer is invalid.

SCORE RULES:

FACTUAL FAITHFULNESS:
1 = incorrect
5 = fully consistent (no hallucination, no missing facts)

SEMANTIC FAITHFULNESS:
1 = different meaning
5 = same meaning

OUTPUT RULES:
- Integers from 1 to 5
- No explanation
- No extra text
- No repetition
- ONLY [[X, Y]]

SOURCE:
{source}

REWRITE:
{rewrite}

FINAL ANSWER:
"""
    return llm(prompt)

In [11]:
# def response_cleaner(response):
#     if isinstance(response, dict):
#         return response

#     if not isinstance(response, str):
#         return None

#     # 1. extraction du bloc JSON
#     match = re.search(r"\{.*\}", response, re.DOTALL)
#     if not match:
#         return None

#     raw = match.group(0)

#     # 2. tentative JSON standard
#     try:
#         return json.loads(raw)
#     except:
#         pass

#     # 3. fallback : Python dict style
#     try:
#         return ast.literal_eval(raw)
#     except:
#         return None

In [12]:
def response_cleaner(response):
   try:
       
       matches = re.findall(r"\[\[\s*(\d)\s*,\s*(\d)\s*\]\]", response)
       if matches:
           # prend la dernière paire
           last = matches[-1]
           return [int(last[0]), int(last[1])]
   except Exception as e:
       print("Parsing error:", e)
   return [0, 0]

In [13]:
cpt = 0
for article in data:
    cpt +=1
    source = article.get("Paragraphes")
    for i in range(1,4):
        rewrite = article.get(f"Rew{i}")
        judge_result = evaluation(source, rewrite)
        article[f"eval{i}"] = judge_result
        try:
            clean_result = response_cleaner(judge_result)
            if clean_result:
                try:
                    article[f"sem_score{i}"] = clean_result[1]
                    article[f"fact_score{i}"] = clean_result[0]
                    print(f"\n {clean_result[0]} \t {clean_result[1]}")
                except Exception as e:
                    print(e)
                    article[f"sem_score{i}"] = -1
                    article[f"fact_score{i}"] = -1
        except Exception as e:
            print(e)
            article[f"sem_score{i}"] = 0
            article[f"fact_score{i}"] = 0
    if cpt % 10 == 0:
        print(f"======    {cpt} ème paragraphes   ==========")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



 4 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 4 	 5

 5 	 5

 4 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 5 	 5

 4 	 5

 5 	 5

 5 	 5

 4 	 5

 5 	 5

 5 	 5

 4 	 5

 4 	 4

 5 	 5

 5 	 5

 5 	 5

 5 	 5
======    10 ème paragraphes   ==========

 5 	 5

 5 	 5

 5 	 5

 4 	 5

 5 	 5

 5 	 4

 5 	 5

 5 	 5

 5 	 5

 4 	 5

 5 	 4

 5 	 4

 5 	 5

 5 	 5

 5 	 5

 4 	 5

 4 	 5

 5 	 5

 5 	 5

 5 	 4

 4 	 5

 4 	 5

 5 	 5

 4 	 5

 4 	 5

 5 	 5

 5 	 4

 5 	 5

 5 	 5

 5 	 5
======    20 ème paragraphes   ==========

 4 	 5

 1 	 1

 4 	 5

 5 	 5

 5 	 5

 4 	 5

 5 	 5

 5 	 4

 5 	 5

 4 	 5

 5 	 5

 5 	 5

 4 	 4

 5 	 5

 5 	 4

 3 	 5

 5 	 5

 5 	 4

 4 	 5

 5 	 5

 5 	 5

 4 	 5

 4 	 5

 4 	 5

 5 	 5

 5 	 5

 2 	 5

 1 	 1

 4 	 5

 5 	 5
======    30 ème paragraphes   ==========

 4 	 4

 4 	 4

 4 	 4

 3 	 4

 5 	 5

 5 	 4

 5 	 5

 4 	 4

 4 	 4

 4 	 5

 5 	 4

 5 	 5

 4 	 5

 5 	 5

 5 	 4

 4 	 5

 4 	 5

 5 	 5

 4 	 5

 

In [14]:
filename = "../data/json/llm_judgment3.json"
if not os.path.exists(filename):
    with open(filename, "w", encoding="utf-8") as f:
        pass

In [15]:
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)